# FraudShield BR - EDA das tres bases

Exploracao rapida das amostras de `data/samples/` antes de treinar qualquer modelo.
Objetivo: ver o desbalanceamento, entender quais sinais separam fraude de legitimo
e checar a sobreposicao entre as classes (o que o modelo NAO vai conseguir separar).

> Uso defensivo. Dados sinteticos nas amostras; para numeros reais use os datasets
> publicos (creditcard.csv da ULB, Enron + Nazario, Cresci et al.).


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

SAMPLES = ROOT / "data" / "samples"
tx = pd.read_csv(SAMPLES / "transacoes_amostra.csv")
mail = pd.read_csv(SAMPLES / "emails_amostra.csv")
social = pd.read_csv(SAMPLES / "social_amostra.csv")
print(tx.shape, mail.shape, social.shape)

## 1. Desbalanceamento: por que accuracy engana

Um modelo que responde sempre "legitimo" ja acerta a taxa da classe majoritaria.

In [ ]:
for name, frame, label in [("transacoes", tx, "Class"), ("emails", mail, "label"), ("social", social, "label")]:
    rate = frame[label].mean()
    print(f"{name:11} n={len(frame):5}  fraude={frame[label].sum():4} ({rate*100:5.2f}%)  "
          f"accuracy do chute majoritario = {max(rate, 1-rate)*100:6.2f}%  recall desse chute = 0%")

## 2. Transacoes: valor, horario e velocidade

In [ ]:
tx["hour"] = (tx.Time / 3600) % 24
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for label, name in [(0, "legitima"), (1, "fraude")]:
    subset = tx[tx.Class == label]
    axes[0].hist(np.log1p(subset.Amount), bins=40, alpha=0.6, density=True, label=name)
    axes[1].hist(subset.hour, bins=24, alpha=0.6, density=True, label=name)
axes[0].set_title("log(1+valor)"); axes[0].legend()
axes[1].set_title("hora do dia"); axes[1].legend()

fraud_rate = tx.groupby(tx.hour.astype(int)).Class.mean()
axes[2].bar(fraud_rate.index, fraud_rate.values, color="#1f9d55")
axes[2].set_title("taxa de fraude por hora")
plt.tight_layout(); plt.show()

print(tx.groupby("Class")[["Amount", "hour"]].describe().round(2).T)

As componentes PCA V1-V28 sao dados sensiveis mascarados (pratica de compliance).
Vamos ver quais delas realmente separam as classes.

In [ ]:
pca_cols = [c for c in tx.columns if c.startswith("V") and c[1:].isdigit()]
separation = (tx[tx.Class == 1][pca_cols].mean() - tx[tx.Class == 0][pca_cols].mean()).abs().sort_values(ascending=False)
print(separation.head(8).round(3))
separation.head(12).sort_values().plot.barh(figsize=(7, 4), color="#1f9d55",
                                           title="|diferenca de media| entre fraude e legitima")
plt.tight_layout(); plt.show()

## 3. E-mail: gatilhos, cabecalho e idade do dominio

In [ ]:
from src.features.text_features import row_features

feats = pd.DataFrame([row_features(row) for _, row in mail.iterrows()])
feats["label"] = mail.label.values
cols = ["trig_total", "trig_urgencia", "trig_credencial", "n_links", "has_ip_url",
        "has_shortener", "has_suspicious_tld", "brand_in_url_wrong_domain",
        "typosquat_hit", "from_reply_mismatch", "spf_fail", "dkim_fail",
        "domain_is_new", "suspicious_attachment", "caps_ratio"]
print(feats.groupby("label")[cols].mean().round(3).T)

In [ ]:
# sobreposicao: phishing SEM nenhum sinal de URL/gatilho (casos tipo BEC)
silent = feats[(feats.label == 1) & (feats.trig_total == 0) & (feats.n_links == 0) & (feats.spf_fail == 0)]
print(f"phishing sem gatilho, sem link e com SPF valido: {len(silent)} de {int(feats.label.sum())}")
print(mail.loc[silent.index, ["from_addr", "subject"]].head())

## 4. Redes sociais: comportamento da conta e rede

In [ ]:
from src.features.social_features import SocialFeatureSpace

space = SocialFeatureSpace(max_features=50)
_, raw = space.fit_transform(social)
raw["label"] = social.label.values
cols = ["account_age_days", "followers_following_ratio", "posts_per_day", "bio_empty",
        "default_profile_image", "username_digit_ratio", "scam_triggers",
        "duplicate_text_count", "creation_burst_size", "has_shortener"]
print(raw.groupby("label")[cols].median().round(2).T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, name in [(0, "autentico"), (1, "fraudulento")]:
    subset = raw[raw.label == label]
    axes[0].scatter(subset.account_age_days, subset.posts_per_day, s=8, alpha=0.5, label=name)
    axes[1].hist(np.clip(subset.creation_burst_size, 0, 40), bins=20, alpha=0.6, density=True, label=name)
axes[0].set_xlabel("idade da conta (dias)"); axes[0].set_ylabel("posts por dia")
axes[0].set_yscale("log"); axes[0].legend(); axes[0].set_title("atividade x idade")
axes[1].set_title("tamanho do cluster de criacao (DBSCAN)"); axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Treinando pelo proprio pipeline do FraudShield

Nada de reimplementar: o notebook usa as mesmas funcoes da CLI.

In [ ]:
from sklearn.model_selection import train_test_split

from src.models.baseline_sklearn import fit_model, positive_proba
from src.models.explainability import global_importance
from src.modules import get_module
from src.pipeline.evaluate import compute_metrics

module = get_module("transaction")
X, y, names, artifacts, raw_tx = module.build_features(tx, None)
idx_tr, idx_te = train_test_split(np.arange(len(y)), test_size=0.25, random_state=42, stratify=y)
model, kind, smote = fit_model("rf", X[idx_tr], y[idx_tr], seed=42, use_smote=True)
print("SMOTE:", smote)

metrics = compute_metrics(y[idx_te], positive_proba(model, X[idx_te]))
print({k: (round(v, 4) if isinstance(v, float) else v)
       for k, v in metrics.items() if k in ("accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc")})
print("matriz de confusao:", metrics["confusion"])

In [ ]:
importance, method = global_importance(model, X[idx_tr], names, top_k=12)
print("metodo:", method)
for name, value in importance:
    print(f"{value:8.5f}  {name}")

## Leitura final

- A fraude e rara: PR-AUC e matriz de confusao contam a verdade que accuracy esconde.
- Parte das fraudes e propositalmente indistinguivel nas amostras (BEC com SPF valido,
  bot maduro, fraude em horario comercial). Recall 1.0 aqui seria sinal de dado irreal.
- Proximo passo: rodar `python main.py --module transaction --demo` e comparar com o
  relatorio gerado em `reports/transaction/relatorio.md`.
